# Gold Layer Notebook

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Creating Schema for gold tables

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS pyspark_module.gold")

DataFrame[]

## Read Silver tables

In [0]:
transactions_df = spark.table("pyspark_module.silver.transactions")
users_df = spark.table("pyspark_module.silver.users")
cards_df = spark.table("pyspark_module.silver.cards")

## Fraud by day of week

In [0]:
# Which day(s) of the week sees the highest number of fraudulent transactions?
gold_fraud_by_day_of_week = (
    transactions_df
    .filter(F.col("is_fraud") == True)
    .groupBy("day_of_week")
    .agg(
        F.count("*").alias("fraud_transaction_count"),
        F.sum("amount").alias("total_fraud_amount")
    )
    .orderBy(F.desc("fraud_transaction_count"))
)

display(gold_fraud_by_day_of_week)

day_of_week,fraud_transaction_count,total_fraud_amount
Friday,5,383.42999999999995
Thursday,2,-48.45999999999998
Monday,2,11.64
Tuesday,1,8.76
Sunday,1,339.0
Saturday,1,23.1


In [0]:
# write to delta
gold_fraud_by_day_of_week.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_by_day_of_week"
)

## Fraud rate trend over the past month

In [0]:
# What is the trend of fraud rate over the past month?
max_date = transactions_df.agg(F.max("transaction_date")).collect()[0][0]

gold_fraud_rate_past_month = (
    transactions_df
    .filter(F.col("transaction_date") >= F.date_sub(F.lit(max_date), 30))
    .groupBy("transaction_date")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
    .orderBy("transaction_date")
)

display(gold_fraud_rate_past_month)

transaction_date,total_transactions,fraud_transactions,fraud_rate
2010-01-01,3463,1,2.8876696505919725E-4
2010-01-02,2989,0,0.0
2010-01-03,3311,1,3.020235578375113E-4
2010-01-04,3244,2,6.165228113440197E-4
2010-01-05,3330,1,3.003003003003003E-4
2010-01-06,3365,0,0.0
2010-01-07,3346,2,5.977286312014345E-4
2010-01-08,3016,4,0.001326259946949602
2010-01-09,3102,1,3.223726627981947E-4
2010-01-10,880,0,0.0


In [0]:
# write to delta
gold_fraud_rate_past_month.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_rate_past_month"
)

## Top users by fraud count

In [0]:
# Which users have the largest number of flagged transactions?
gold_top_fraud_users = (
    transactions_df
    .filter(F.col("is_fraud") == True)
    .groupBy("client_id")
    .agg(
        F.count("*").alias("fraud_transaction_count"),
        F.sum("amount").alias("total_fraud_amount"),
        F.avg("amount").alias("avg_fraud_amount")
    )
    .join(
        users_df.select(
            "client_id",
            "current_age",
            "gender",
            "yearly_income",
            "credit_score",
            "num_credit_cards"
        ),
        on="client_id",
        how="left"
    )
    .orderBy(F.desc("fraud_transaction_count"))
)

display(gold_top_fraud_users)

client_id,fraud_transaction_count,total_fraud_amount,avg_fraud_amount,current_age,gender,yearly_income,credit_score,num_credit_cards
126,5,329.14000000000004,65.828,63,Male,26600.0,799,4
379,2,28.23,14.115,47,Female,43496.0,765,3
720,2,11.64,5.82,36,Female,19800.0,682,3
1644,2,154.92,77.46,83,Male,31943.0,686,6
1600,1,193.54,193.54,62,Female,101193.0,747,3


In [0]:
# write to delta
gold_top_fraud_users.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.top_fraud_users"
)

## Fraud by card attributes

In [0]:
gold_fraud_by_card_attributes = (
    transactions_df
    .join(
        cards_df.select(
            "card_id",
            "card_type",
            "card_brand",
            "has_chip",
            "card_on_dark_web"
        ),
        on="card_id",
        how="left"
    )
    .groupBy("card_type", "card_brand", "has_chip", "card_on_dark_web")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)).alias("total_fraud_amount"),
        F.avg("amount").alias("avg_transaction_amount")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
    .orderBy(F.desc("fraud_rate"), F.desc("fraud_transactions"))
)

display(gold_fraud_by_card_attributes)

card_type,card_brand,has_chip,card_on_dark_web,total_transactions,fraud_transactions,total_fraud_amount,avg_transaction_amount,fraud_rate
Debit,Mastercard,YES,No,10530,10,562.5500000000001,40.34478632478639,9.49667616334283E-4
Credit,Visa,YES,No,3213,2,154.92,53.81361344537817,6.224712107065049E-4
Debit,Visa,YES,No,6362,0,0.0,41.38334328827417,0.0
Credit,Mastercard,NO,No,343,0,0.0,54.89349854227403,0.0
Debit,Mastercard,NO,No,1225,0,0.0,38.400661224489795,0.0
Credit,Amex,NO,No,182,0,0.0,70.47620879120872,0.0
Debit (Prepaid),Visa,YES,No,494,0,0.0,20.208299595141707,0.0
Credit,Mastercard,YES,No,2596,0,0.0,49.8504506933744,0.0
Debit,Visa,NO,No,482,0,0.0,33.72255186721992,0.0
Debit (Prepaid),Mastercard,NO,No,128,0,0.0,21.474765625,0.0


In [0]:
gold_fraud_by_card_attributes.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_by_card_attributes"
)

## User spending spikes versus weekly average

In [0]:
# Are there any users showing a sharp rise in transaction amount compared to their weekly average?
user_weekly_avg_df = (
    transactions_df
    .groupBy("client_id", "transaction_week")
    .agg(
        F.avg("amount").alias("weekly_avg_amount")
    )
)

gold_user_amount_spikes = (
    transactions_df
    .join(user_weekly_avg_df, on=["client_id", "transaction_week"], how="left")
    .withColumn(
        "spike_ratio",
        F.when(F.col("weekly_avg_amount") > 0, F.col("amount") / F.col("weekly_avg_amount"))
         .otherwise(None)
    )
    .filter(F.col("spike_ratio") >= 3)
    .select(
        "transaction_id",
        "client_id",
        "transaction_date",
        "transaction_week",
        "amount",
        "weekly_avg_amount",
        "spike_ratio",
        "is_fraud",
        "merchant_id",
        "mcc",
        "mcc_description"
    )
    .orderBy(F.desc("spike_ratio"))
)

display(gold_user_amount_spikes)

transaction_id,client_id,transaction_date,transaction_week,amount,weekly_avg_amount,spike_ratio,is_fraud,merchant_id,mcc,mcc_description
7482722,900,2010-01-02,53,207.13,0.44363636363636066,466.89139344262605,false,75062,4900,"Utilities - Electric, Gas, Water, Sanitary"
7477361,900,2010-01-01,53,175.9,0.44363636363636066,396.4959016393469,false,16790,3389,Non-Precious Metal Services
7477434,900,2010-01-01,53,87.0,0.44363636363636066,196.1065573770505,false,61195,5541,Service Stations
7477545,900,2010-01-01,53,52.1,0.44363636363636066,117.43852459016472,false,61195,5541,Service Stations
7503036,216,2010-01-08,1,314.0,3.277857142857145,95.7942906951405,false,7777,3684,Semiconductors and Related Devices
7479463,900,2010-01-02,53,38.71,0.44363636363636066,87.25614754098419,false,36500,5812,Eating Places and Restaurants
7498991,383,2010-01-07,1,121.84,1.988666666666667,61.26718069057994,false,74934,3596,Miscellaneous Machinery and Parts Manufacturing
7492932,383,2010-01-05,1,113.95,1.988666666666667,57.29969829031176,false,98787,7996,"Amusement Parks, Carnivals, Circuses"
7476725,549,2010-01-01,53,327.53,6.089999999999992,53.78160919540237,false,39991,3771,Railroad Passenger Transport
7478252,122,2010-01-01,53,203.29,3.796923076923076,53.540721231766625,false,44795,3780,Computer Network Services


In [0]:
# write to delta
gold_user_amount_spikes.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.user_amount_spikes"
)

## Fraud rate by merchant category

In [0]:
# Which merchant categories exhibit the highest fraud rate?
gold_fraud_rate_by_merchant_category = (
    transactions_df
    .groupBy("mcc", "mcc_description")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)).alias("total_fraud_amount")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
    .orderBy(F.desc("fraud_rate"))
)

display(gold_fraud_rate_by_merchant_category)

mcc,mcc_description,total_transactions,fraud_transactions,total_fraud_amount,fraud_rate
5712,"Furniture, Home Furnishings, and Equipment Stores",8,1,8.76,0.125
3640,"Lighting, Fixtures, Electrical Supplies",30,3,290.54,0.1
4722,Travel Agencies,60,1,0.19,0.016666666666666666
5815,"Digital Goods - Media, Books, Apps",155,1,5.13,0.0064516129032258064
4214,Motor Freight Carriers and Trucking,231,1,4.45,0.004329004329004329
5311,Department Stores,1046,3,176.45,0.0028680688336520078
5814,Fast Food Restaurants,1156,1,38.41,8.650519031141869E-4
5300,Wholesale Clubs,1392,1,193.54,7.183908045977011E-4
9402,Postal Services - Government Only,179,0,0.0,0.0
7549,Towing Services,4,0,0.0,0.0


In [0]:
# write to delta
gold_fraud_rate_by_merchant_category.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_rate_by_merchant_category"
)

## Merchants with unusually high fraud volume

In [0]:
# Are there specific merchants with unusually high fraud volume?
merchant_fraud_df = (
    transactions_df
    .groupBy("merchant_id")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)).alias("total_fraud_amount")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
)

fraud_stats = merchant_fraud_df.agg(
    F.avg("fraud_transactions").alias("avg_fraud_count"),
    F.stddev("fraud_transactions").alias("std_fraud_count")
).collect()[0]

avg_fraud_count = fraud_stats["avg_fraud_count"]
std_fraud_count = fraud_stats["std_fraud_count"]

gold_high_fraud_merchants = (
    merchant_fraud_df
    .withColumn(
        "is_unusually_high_fraud",
        F.col("fraud_transactions") > F.lit(avg_fraud_count + (2 * std_fraud_count))
    )
    .filter(F.col("is_unusually_high_fraud") == True)
    .orderBy(F.desc("fraud_transactions"))
)

display(gold_high_fraud_merchants)

merchant_id,total_transactions,fraud_transactions,total_fraud_amount,fraud_rate,is_unusually_high_fraud
3558,30,3,290.54,0.1,true
38602,42,1,7.19,0.023809523809523808,true
59199,12,1,38.41,0.08333333333333333,true
90999,20,1,0.19,0.05,true
54773,9,1,146.16,0.1111111111111111,true
47399,57,1,5.13,0.017543859649122806,true
81477,25,1,23.1,0.04,true
24504,130,1,4.45,0.007692307692307693,true
60569,704,1,193.54,0.0014204545454545455,true
21776,1,1,8.76,1.0,true


In [0]:
# write to delta
gold_high_fraud_merchants.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.high_fraud_merchants"
)

## Fraud distribution by time of day

In [0]:
# How does fraud distribution vary by time of day?
gold_fraud_by_time_of_day = (
    transactions_df
    .groupBy("time_of_day")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)).alias("total_fraud_amount")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
    .orderBy(F.desc("fraud_transactions"))
)

display(gold_fraud_by_time_of_day)

time_of_day,total_transactions,fraud_transactions,total_fraud_amount,fraud_rate
Night,2748,7,194.98,0.002547307132459971
Evening,5107,5,522.49,9.790483649892304E-4
Afternoon,9878,0,0.0,0.0
Morning,12313,0,0.0,0.0


In [0]:
# write to delta
gold_fraud_by_time_of_day.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_by_time_of_day"
)

## Average amount for fraud vs non-fraud

In [0]:
# What’s the average transaction amount for fraud vs non-fraud transactions?
gold_avg_amount_fraud_vs_nonfraud = (
    transactions_df
    .groupBy("is_fraud")
    .agg(
        F.count("*").alias("transaction_count"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.min("amount").alias("min_transaction_amount"),
        F.max("amount").alias("max_transaction_amount"),
        F.sum("amount").alias("total_transaction_amount")
    )
    .orderBy("is_fraud")
)

display(gold_avg_amount_fraud_vs_nonfraud)

is_fraud,transaction_count,avg_transaction_amount,min_transaction_amount,max_transaction_amount,total_transaction_amount
false,30034,42.613735433175385,-500.0,1812.76,1279860.9299999895
true,12,59.78916666666666,-339.0,339.0,717.4699999999999


In [0]:
# write to delta
gold_avg_amount_fraud_vs_nonfraud.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.avg_amount_fraud_vs_nonfraud"
)

## Total fraud amount by merchant category

In [0]:
# Which merchant category has the highest total fraud amount?
gold_fraud_amount_by_merchant_category = (
    transactions_df
    .filter(F.col("is_fraud") == True)
    .groupBy("mcc", "mcc_description")
    .agg(
        F.count("*").alias("fraud_transactions"),
        F.sum("amount").alias("total_fraud_amount"),
        F.avg("amount").alias("avg_fraud_amount")
    )
    .orderBy(F.desc("total_fraud_amount"))
)

display(gold_fraud_amount_by_merchant_category)

mcc,mcc_description,fraud_transactions,total_fraud_amount,avg_fraud_amount
3640,"Lighting, Fixtures, Electrical Supplies",3,290.54,96.84666666666668
5300,Wholesale Clubs,1,193.54,193.54
5311,Department Stores,3,176.45,58.81666666666666
5814,Fast Food Restaurants,1,38.41,38.41
5712,"Furniture, Home Furnishings, and Equipment Stores",1,8.76,8.76
5815,"Digital Goods - Media, Books, Apps",1,5.13,5.13
4214,Motor Freight Carriers and Trucking,1,4.45,4.45
4722,Travel Agencies,1,0.19,0.19


In [0]:
# write to delta
gold_fraud_amount_by_merchant_category.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_amount_by_merchant_category"
)

## Daily fraud losses

In [0]:
# What are the total monetary losses due to fraud each day?
gold_daily_fraud_losses = (
    transactions_df
    .filter(F.col("is_fraud") == True)
    .groupBy("transaction_date")
    .agg(
        F.count("*").alias("fraud_transactions"),
        F.sum("amount").alias("daily_fraud_loss")
    )
    .orderBy("transaction_date")
)

display(gold_daily_fraud_losses)

transaction_date,fraud_transactions,daily_fraud_loss
2010-01-01,1,0.19
2010-01-03,1,339.0
2010-01-04,2,11.64
2010-01-05,1,8.76
2010-01-07,2,-48.45999999999998
2010-01-08,4,383.24
2010-01-09,1,23.1


In [0]:
# write to delta
gold_daily_fraud_losses.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.daily_fraud_losses"
)

## Weekly unique fraud users

In [0]:
# How many unique users commit fraudulent transactions per week?
gold_weekly_unique_fraud_users = (
    transactions_df
    .filter(F.col("is_fraud") == True)
    .groupBy("transaction_week")
    .agg(
        F.countDistinct("client_id").alias("unique_fraud_users"),
        F.count("*").alias("fraud_transactions"),
        F.sum("amount").alias("weekly_fraud_amount")
    )
    .orderBy("transaction_week")
)

display(gold_weekly_unique_fraud_users)

transaction_week,unique_fraud_users,fraud_transactions,weekly_fraud_amount
1,5,10,378.28
53,1,2,339.19


In [0]:
# write to delta
gold_weekly_unique_fraud_users.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.weekly_unique_fraud_users"
)

## Monthly fraud trend

In [0]:
# Do fraud patterns show seasonal or monthly spikes?
gold_monthly_fraud_trend = (
    transactions_df
    .groupBy("transaction_month")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)).alias("monthly_fraud_amount")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
    .orderBy("transaction_month")
)

display(gold_monthly_fraud_trend)

transaction_month,total_transactions,fraud_transactions,monthly_fraud_amount,fraud_rate
2010-01,30046,12,717.4699999999999,3.99387605671304E-4


In [0]:
# write to delta
gold_monthly_fraud_trend.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.monthly_fraud_trend"
)

## User behavior before vs after first fraud event



In [0]:
# How has user behavior changed before versus after a fraudulent event?
first_fraud_df = (
    transactions_df
    .filter(F.col("is_fraud") == True)
    .groupBy("client_id")
    .agg(F.min("transaction_date").alias("first_fraud_date"))
)

transactions_with_fraud_event_df = (
    transactions_df
    .join(first_fraud_df, on="client_id", how="inner")
    .withColumn(
        "fraud_event_period",
        F.when(F.col("transaction_date") < F.col("first_fraud_date"), "Before Fraud")
         .when(F.col("transaction_date") > F.col("first_fraud_date"), "After Fraud")
         .otherwise("Fraud Day")
    )
)

gold_user_behavior_before_after_fraud = (
    transactions_with_fraud_event_df
    .groupBy("client_id", "fraud_event_period")
    .agg(
        F.count("*").alias("transaction_count"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.sum("amount").alias("total_transaction_amount"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .orderBy("client_id", "fraud_event_period")
)

display(gold_user_behavior_before_after_fraud)

client_id,fraud_event_period,transaction_count,avg_transaction_amount,total_transaction_amount,fraud_transactions
126,After Fraud,42,30.641428571428573,1286.94,4
126,Fraud Day,7,16.535714285714285,115.75,1
379,After Fraud,8,10.594999999999999,84.75999999999999,1
379,Before Fraud,46,22.465652173913046,1033.42,0
379,Fraud Day,6,28.298333333333336,169.79000000000002,1
720,After Fraud,1,32.53,32.53,0
720,Before Fraud,4,38.825,155.3,0
720,Fraud Day,3,11.866666666666667,35.6,2
1600,After Fraud,2,15.705,31.41,0
1600,Before Fraud,21,96.34047619047618,2023.1499999999999,0


In [0]:
# write to delta
gold_user_behavior_before_after_fraud.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.user_behavior_before_after_fraud"
)

## Fraud by transaction amount bucket

In [0]:
# Are fraudulent transactions more common on high-value purchases compared to low-value purchases
gold_fraud_by_amount_bucket = (
    transactions_df
    .withColumn(
        "amount_bucket",
        F.when(F.col("amount") < 25, "Low: < $25")
         .when(F.col("amount").between(25, 100), "Medium: $25-$100")
         .when(F.col("amount").between(100, 500), "High: $100-$500")
         .otherwise("Very High: $500+")
    )
    .groupBy("amount_bucket")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.sum(F.when(F.col("is_fraud") == True, F.col("amount")).otherwise(0)).alias("total_fraud_amount")
    )
    .withColumn("fraud_rate", F.col("fraud_transactions") / F.col("total_transactions"))
    .orderBy(F.desc("fraud_rate"))
)

display(gold_fraud_by_amount_bucket)

amount_bucket,total_transactions,fraud_transactions,avg_transaction_amount,total_fraud_amount,fraud_rate
High: $100-$500,3107,4,165.20349211458006,969.24,0.001287415513356936
Low: < $25,13961,7,-2.332752668146982,-290.17999999999995,5.013967480839482E-4
Medium: $25-$100,12871,1,55.625337580607486,38.41,7.76940408670655E-5
Very High: $500+,107,0,784.1587850467288,0.0,0.0


In [0]:
# write to delta
gold_fraud_by_amount_bucket.write.mode("overwrite").format("delta").saveAsTable(
    "pyspark_module.gold.fraud_by_amount_bucket"
)